In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold, ParameterGrid
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector
import importlib

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
eg_subject = 'SUBJ_040724'
single_sequence = 'SEQ_065189'
dc_offset_max = 10
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
log_edges = np.logspace(np.log10(0.5), np.log10(100), num=15)
sequences = ['SEQ_000034', 'SEQ_065478', 'SEQ_065470']
subjects = ['SUBJ_059520']
band_edges = np.arange(0, 101, 10)
pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
subject_cols = ['orientation', 'subject', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000016', 'SEQ_000018',
                    'SEQ_000022', 'SEQ_000033', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053',
                    'SEQ_000058', 'SEQ_000063', 'SEQ_000079', 'SEQ_000091', 'SEQ_000092',
                    'SEQ_000111', 'SEQ_000113', 'SEQ_000114', 'SEQ_000142', 'SEQ_000150']
some_subjects = ['SUBJ_059520', 'SUBJ_020948', 'SUBJ_040282', 'SUBJ_052342', 'SUBJ_032165',
                'SUBJ_024086', 'SUBJ_040733', 'SUBJ_063346', 'SUBJ_055211', 'SUBJ_001430',
                'SUBJ_012088', 'SUBJ_040310', 'SUBJ_032233', 'SUBJ_059330', 'SUBJ_013623',
                'SUBJ_032585', 'SUBJ_063464', 'SUBJ_038023', 'SUBJ_044680', 'SUBJ_024137']

In [ ]:
train_df = raw_train_df.set_index('row_id')

single_subject_df = train_df.loc[train_df['subject'] == eg_subject, :]

single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

multiple_gesture_df = train_df.loc[train_df['sequence_id'].isin(some_sequences), :]

some_subjects_df = train_df.loc[train_df['subject'].isin(some_subjects), :]

master_test_df = raw_test_df.merge(test_demo_df, on='subject', how='left').set_index('row_id')

In [ ]:
# 0.47 across all domains

importlib.reload(utils)

num_pattern = 'acc|rot|thm'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols = ['orientation', 'subject']
normal_cols = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

sliced_data_df = train_df.copy(deep=True)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            OrdinalEncoder(),
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

custom_extractor = utils.ImuExtractor(
    subject_df=train_demo_df,
    imu_sensor_list=['acc_x'],
    rotation_sensor_list=['rot_x', 'rot_y', 'rot_z'],
    disable_tqdm=False
)

preprocessor.set_output(transform='pandas')

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier(), custom_extractor))
], memory='cache_dir')

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x', 'acc_y', 'acc_z']],
    f'{pipe_name}__imu_domain': ['acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes': [True],
    f'{pipe_name}__sampling_rate': [100],

    f'{pipe_name}__rotation_sensor_list': [['rot_w','rot_x', 'rot_y', 'rot_z']],
    f'{pipe_name}__combine_rot_axes': [True],
    f'{pipe_name}__rotation_domain': ['acceleration'],

    f'{pipe_name}__thermopile_sensor_list': [['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']],

    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None],

    f'{pipe_name}__category_data': [True],
    
    f'{pipe_name}__segmentation': [None]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=3),
    verbose=1,
    n_jobs=1
)

y = sliced_data_df[['sequence_id', 'gesture']]

grid_search.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

pd.DataFrame(grid_search.cv_results_)

In [ ]:
# Get best model
best_model = grid_search.best_estimator_

# Get feature names from preprocessor
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# Get classifier and importances
rf = best_model.named_steps['classifier'].estimator_
importances = rf.feature_importances_

# Create DataFrame
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

feat_imp_df['items'] = feat_imp_df['feature'].str.split('_')
feat_imp_df['size_items'] = feat_imp_df['items'].apply(len)
feat_imp_df['name'] = feat_imp_df['items'].str[0]

feat_imp_df['percentage'] = feat_imp_df['importance'] / feat_imp_df['importance'].sum() * 100
feat_imp_df['created_feature'] = feat_imp_df['items'].str[-1]

feat_imp_df


In [ ]:
feat_imp_df.groupby(['name', 'created_feature'])['percentage'].sum().sort_values(ascending=False)

In [ ]:
feat_imp_df.groupby('created_feature')['percentage'].sum().sort_values(ascending=False)

In [ ]:
# Plot top 20
plt.figure(figsize=(10, 8))
plt.barh(feat_imp_df['feature'].head(20), feat_imp_df['importance'].head(20))
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
importlib.reload(utils)

extractor = utils.ImuExtractor(
                imu_sensor_list=['acc_x'],
                category_data=True,
                segmentation='both'
                                   )

pipeline = Pipeline([
    (pipe_name, utils.ImuExtractor(
                              #      imu_sensor_list=['acc_x'],
                              #   rotation_sensor_list=['rot_x', 'rot_y'],
                              thermopile_sensor_list=['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'],
                                   category_data=True,
                                #    segmentation='both'
                                segmentation=None,
                                combine_rot_axes = True
                                   )),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier(), extractor))
])
pipeline['preprocessor'].set_output(transform='pandas')

X_eg = pipeline[:1].fit_transform(single_gesture_df)
y_eg = single_gesture_df[['sequence_id', 'gesture']]
X_eg